In [8]:
import sys
import warnings
sys.path.append('..')

import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from src.config import TRAIN_PATH, TEST_PATH
from src.preprocessing import HousePricesPreprocessor
from src.validation import cross_validate

warnings.filterwarnings('ignore')

In [9]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [10]:
y = np.log1p(train['SalePrice'])
test_ids = test['Id']
train = train.drop(['SalePrice', 'Id'], axis=1)
test = test.drop('Id', axis=1)

# Preprocessing
prep_scaled = HousePricesPreprocessor(scale=True)
X_scaled = prep_scaled.fit_transform(train, np.expm1(y))

prep_raw = HousePricesPreprocessor(scale=False)
X_raw = prep_raw.fit_transform(train, np.expm1(y))

# Выбросы
outlier_idx = X_raw[(X_raw['GrLivArea'] > 4000) & (np.expm1(y) < 200000)].index
X_scaled = X_scaled.drop(outlier_idx)
X_raw = X_raw.drop(outlier_idx)
y = y.drop(outlier_idx)

print(f"X_scaled: {X_scaled.shape}")
print(f"X_raw: {X_raw.shape}")

X_scaled: (1458, 99)
X_raw: (1458, 99)


KNN

In [11]:
for n in [3, 5, 10, 15, 20]:
    print(f"n_neighbors={n}:")
    cross_validate(KNeighborsRegressor(n_neighbors=n), X_scaled, y)

n_neighbors=3:
RMSE по фолдам: [0.188  0.1771 0.1932 0.1881 0.2154]
Среднее: 0.1923 ± 0.0127
n_neighbors=5:
RMSE по фолдам: [0.1825 0.1722 0.1848 0.1818 0.2068]
Среднее: 0.1856 ± 0.0115
n_neighbors=10:
RMSE по фолдам: [0.1791 0.1721 0.1774 0.1771 0.1994]
Среднее: 0.1810 ± 0.0095
n_neighbors=15:
RMSE по фолдам: [0.1806 0.1708 0.1776 0.1791 0.2027]
Среднее: 0.1822 ± 0.0108
n_neighbors=20:
RMSE по фолдам: [0.1813 0.1752 0.1804 0.1795 0.204 ]
Среднее: 0.1841 ± 0.0102


KNN: weights и metric (n=10)

In [12]:
for weight in ['uniform', 'distance']:
    for metric in ['euclidean', 'manhattan']:
        print(f"weights={weight}, metric={metric}:")
        cross_validate(KNeighborsRegressor(n_neighbors=10, weights=weight, metric=metric), X_scaled, y)

weights=uniform, metric=euclidean:
RMSE по фолдам: [0.1791 0.1721 0.1774 0.1771 0.1994]
Среднее: 0.1810 ± 0.0095
weights=uniform, metric=manhattan:
RMSE по фолдам: [0.1738 0.1595 0.169  0.1733 0.1777]
Среднее: 0.1707 ± 0.0062
weights=distance, metric=euclidean:
RMSE по фолдам: [0.1775 0.1707 0.1762 0.1758 0.1991]
Среднее: 0.1799 ± 0.0099
weights=distance, metric=manhattan:
RMSE по фолдам: [0.1715 0.1571 0.1669 0.171  0.1755]
Среднее: 0.1684 ± 0.0063


Decision Tree

In [13]:
for depth in [3, 5, 10, 15, 20, None]:
    print(f"max_depth={depth}:")
    cross_validate(DecisionTreeRegressor(max_depth=depth, random_state=255), X_raw, y)

max_depth=3:
RMSE по фолдам: [0.221  0.2184 0.2126 0.2363 0.2131]
Среднее: 0.2203 ± 0.0086
max_depth=5:
RMSE по фолдам: [0.1866 0.1891 0.1877 0.2007 0.1949]
Среднее: 0.1918 ± 0.0053
max_depth=10:
RMSE по фолдам: [0.1955 0.1961 0.1902 0.2062 0.193 ]
Среднее: 0.1962 ± 0.0054
max_depth=15:
RMSE по фолдам: [0.1944 0.2096 0.1965 0.213  0.1968]
Среднее: 0.2021 ± 0.0077
max_depth=20:
RMSE по фолдам: [0.1926 0.1958 0.1869 0.2092 0.1993]
Среднее: 0.1967 ± 0.0074
max_depth=None:
RMSE по фолдам: [0.2101 0.194  0.194  0.2106 0.1967]
Среднее: 0.2011 ± 0.0076


Random Forest

In [14]:
for n_estimators in [100, 300, 500]:
    for max_depth in [10, 15, 20, None]:
        print(f"n_estimators={n_estimators}, max_depth={max_depth}:")
        cross_validate(RandomForestRegressor(
            n_estimators=n_estimators, max_depth=max_depth, random_state=255
        ), X_raw, y)

n_estimators=100, max_depth=10:
RMSE по фолдам: [0.1326 0.1471 0.1304 0.1488 0.129 ]
Среднее: 0.1376 ± 0.0086
n_estimators=100, max_depth=15:
RMSE по фолдам: [0.132  0.1472 0.1286 0.1485 0.1284]
Среднее: 0.1369 ± 0.0090
n_estimators=100, max_depth=20:
RMSE по фолдам: [0.1321 0.1466 0.1291 0.1483 0.1277]
Среднее: 0.1368 ± 0.0089
n_estimators=100, max_depth=None:
RMSE по фолдам: [0.132  0.1467 0.1292 0.149  0.1282]
Среднее: 0.1370 ± 0.0090
n_estimators=300, max_depth=10:
RMSE по фолдам: [0.1343 0.1474 0.1281 0.1479 0.1285]
Среднее: 0.1372 ± 0.0088
n_estimators=300, max_depth=15:
RMSE по фолдам: [0.1339 0.147  0.1266 0.147  0.1279]
Среднее: 0.1365 ± 0.0089
n_estimators=300, max_depth=20:
RMSE по фолдам: [0.1338 0.1466 0.1272 0.1471 0.1275]
Среднее: 0.1364 ± 0.0088
n_estimators=300, max_depth=None:
RMSE по фолдам: [0.1337 0.1466 0.1273 0.1474 0.1275]
Среднее: 0.1365 ± 0.0089
n_estimators=500, max_depth=10:
RMSE по фолдам: [0.1338 0.1472 0.1281 0.1474 0.1279]
Среднее: 0.1369 ± 0.0088
n_esti

Итоги: KNN, Tree, Random Forest

- KNN (n=10, distance, manhattan):     CV=0.1684
- Decision Tree (max_depth=5):         CV=0.1918
- Random Forest (500, depth=15):       CV=0.1361